# Quantum Concrete Crack Classification

This notebook follows a similar staged workflow to the YOLO notebook, but uses a **hybrid Quantum CNN** for crack classification.

Pipeline: environment checks → data preparation → hybrid quantum model setup → training → evaluation → final summary.

## 1. Environment Setup
Use the dedicated environment created for this stage: `quantum_cnn/.venv`.

In [ ]:
import os
import sys
import time
import random
from pathlib import Path

import numpy as np
import torch
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.metrics import classification_report, confusion_matrix, roc_auc_score

plt.style.use('seaborn-v0_8-darkgrid')
sns.set_palette('husl')

print('Python:', sys.executable)
print('PyTorch:', torch.__version__)
print('NumPy:', np.__version__)

In [ ]:
RANDOM_SEED = 42
random.seed(RANDOM_SEED)
np.random.seed(RANDOM_SEED)
torch.manual_seed(RANDOM_SEED)

if torch.cuda.is_available():
    device = 'cuda'
    print(f'GPU detected: {torch.cuda.get_device_name(0)}')
else:
    device = 'cpu'
    print('No GPU detected, using CPU.')

print('Device:', device)

## 2. Data Preparation

In [ ]:
cwd = Path.cwd().resolve()
if (cwd / 'src').exists():
    PROJECT_ROOT = cwd
elif (cwd.parent / 'src').exists():
    PROJECT_ROOT = cwd.parent
else:
    PROJECT_ROOT = cwd

SRC_PATH = PROJECT_ROOT / 'src'
if str(SRC_PATH) not in sys.path:
    sys.path.append(str(SRC_PATH))

DATASET_ROOT = PROJECT_ROOT / 'raw_classification'
print('Project root:', PROJECT_ROOT)
print('Dataset root:', DATASET_ROOT)

In [ ]:
if not DATASET_ROOT.exists():
    raise FileNotFoundError(f'Dataset directory not found: {DATASET_ROOT}')

class_dirs = sorted([d for d in DATASET_ROOT.iterdir() if d.is_dir()])
if len(class_dirs) < 2:
    raise RuntimeError('Expected at least two class folders under raw_classification/')

print('Detected classes:')
total_images = 0
for class_dir in class_dirs:
    count = len(list(class_dir.glob('*.jpg')) + list(class_dir.glob('*.jpeg')) + list(class_dir.glob('*.png')))
    total_images += count
    print(f'  - {class_dir.name}: {count}')

print(f'Total images: {total_images}')

In [ ]:
import cv2

sample_paths = []
for class_dir in class_dirs:
    candidates = sorted(list(class_dir.glob('*.jpg')) + list(class_dir.glob('*.jpeg')) + list(class_dir.glob('*.png')))
    sample_paths.extend(candidates[:2])

show_n = min(6, len(sample_paths))
plt.figure(figsize=(12, 6))
for i, img_path in enumerate(sample_paths[:show_n], 1):
    img_bgr = cv2.imread(str(img_path))
    img_rgb = cv2.cvtColor(img_bgr, cv2.COLOR_BGR2RGB)
    plt.subplot(2, 3, i)
    plt.imshow(img_rgb)
    plt.title(img_path.parent.name)
    plt.axis('off')
plt.tight_layout()
plt.show()

## 3. Quantum CNN Configuration

In [ ]:
from quantum_cnn.config import QuantumCNNConfig

config = QuantumCNNConfig(
    dataset_root=str(DATASET_ROOT),
    image_size=32,
    batch_size=32,
    train_ratio=0.70,
    val_ratio=0.15,
    test_ratio=0.15,
    learning_rate=1e-3,
    weight_decay=1e-4,
    epochs=8,
    seed=RANDOM_SEED,
    n_qubits=4,
    n_q_layers=2,
    quantum_embedding_dim=8,
    hidden_dim=64,
    output_root=str(PROJECT_ROOT / 'runs' / 'quantum_cnn'),
    experiment_name='notebook_run',
)

print(config)

## 4. Build Dataloaders and Hybrid Quantum Model

In [ ]:
from quantum_cnn.data import create_dataloaders
from quantum_cnn.hybrid_qcnn import HybridQuantumCNN

train_loader, val_loader, test_loader, classes = create_dataloaders(config)

model = HybridQuantumCNN(
    num_classes=len(classes),
    n_qubits=config.n_qubits,
    n_q_layers=config.n_q_layers,
    quantum_embedding_dim=config.quantum_embedding_dim,
    hidden_dim=config.hidden_dim,
)
model = model.to(device)

print('Classes:', classes)
print('Train batches:', len(train_loader))
print('Val   batches:', len(val_loader))
print('Test  batches:', len(test_loader))
print(model)

## 5. Train the Hybrid Quantum CNN

In [ ]:
from quantum_cnn.train_quantum_cnn import train_model

train_start = time.time()
history, best_model = train_model(
    model=model,
    train_loader=train_loader,
    val_loader=val_loader,
    epochs=config.epochs,
    learning_rate=config.learning_rate,
    weight_decay=config.weight_decay,
    device=device,
)
train_time = time.time() - train_start
print(f'\nTraining complete in {train_time/60:.2f} minutes')

## 6. Evaluate on Test Set

In [ ]:
best_model.eval()
all_preds = []
all_labels = []
all_probs = []

with torch.no_grad():
    for images, labels in test_loader:
        images = images.to(device)
        labels = labels.to(device)
        logits = best_model(images)
        probs = torch.softmax(logits, dim=1)
        preds = probs.argmax(dim=1)

        all_preds.extend(preds.cpu().numpy().tolist())
        all_labels.extend(labels.cpu().numpy().tolist())
        all_probs.extend(probs.cpu().numpy().tolist())

test_acc = float((np.array(all_preds) == np.array(all_labels)).mean())
print(f'Test accuracy: {test_acc:.4f}')
print('\nClassification Report:')
print(classification_report(all_labels, all_preds, target_names=classes, digits=4))

In [ ]:
cm = confusion_matrix(all_labels, all_preds)
plt.figure(figsize=(6, 5))
sns.heatmap(cm, annot=True, fmt='d', cmap='Blues', xticklabels=classes, yticklabels=classes)
plt.title('Confusion Matrix')
plt.xlabel('Predicted')
plt.ylabel('True')
plt.tight_layout()
plt.show()

if len(classes) == 2:
    pos_probs = [row[1] for row in all_probs]
    auc = roc_auc_score(all_labels, pos_probs)
    print(f'ROC-AUC: {auc:.4f}')

In [ ]:
epochs = range(1, len(history['train_loss']) + 1)

plt.figure(figsize=(12, 4))
plt.subplot(1, 2, 1)
plt.plot(epochs, history['train_loss'], marker='o', label='Train Loss')
plt.plot(epochs, history['val_loss'], marker='o', label='Val Loss')
plt.xlabel('Epoch')
plt.ylabel('Loss')
plt.title('Loss Curves')
plt.legend()

plt.subplot(1, 2, 2)
plt.plot(epochs, history['train_acc'], marker='o', label='Train Acc')
plt.plot(epochs, history['val_acc'], marker='o', label='Val Acc')
plt.xlabel('Epoch')
plt.ylabel('Accuracy')
plt.title('Accuracy Curves')
plt.legend()
plt.tight_layout()
plt.show()

## 7. Save Model and Final Summary

In [ ]:
import pathlib as _pl
save_dir = _pl.Path(config.output_root) / config.experiment_name
save_dir.mkdir(parents=True, exist_ok=True)
model_path = save_dir / 'best_hybrid_quantum_cnn.pt'
torch.save(best_model.state_dict(), model_path)

print('=' * 70)
print('QUANTUM CNN CRACK CLASSIFICATION - FINAL SUMMARY')
print('=' * 70)
print(f'Dataset Root : {DATASET_ROOT}')
print(f'Classes      : {classes}')
print(f'Epochs       : {config.epochs}')
print(f'Qubits       : {config.n_qubits}  |  Q-Layers: {config.n_q_layers}')
print(f'Test Accuracy: {test_acc:.4f}')
if len(classes) == 2 and 'auc' in dir():
    print(f'ROC-AUC      : {auc:.4f}')
print(f'Training Time: {train_time/60:.2f} minutes')
print(f'Saved Model  : {model_path}')
print('=' * 70)